# GECS Task 2 — SubIndustry Classification

**Target:** 428 SubIndustry classes | **Data:** 27,537 rows | **Model:** TF-IDF word+char + Logistic Regression

**Why TF-IDF not neural:** Neural models consistently underperform on Task 2 due to extreme class imbalance (54 classes < 5 training examples). Friend's report confirmed TF-IDF wins here.

**Text input:** SegmentName + SegmentDescription only — LongProfile excluded (reduces F1 by 0.089 per friend's findings).

In [1]:
import numpy as np
import pandas as pd
import time
import warnings
import pickle
import json
import copy
from pathlib import Path
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score, classification_report
from scipy.sparse import hstack
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────
# Local:
BASE_DIR  = Path.home() / 'Documents/capstone/depaul-morningstar-capstone'
# Colab: BASE_DIR = Path('/content/drive/MyDrive/CAPSTONE')

RAW_DIR    = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR / 'data' / 'cleaned_task2'
ART_DIR    = BASE_DIR / 'task2_artifacts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

FILE_T2 = RAW_DIR / 'task2_subindustry_classification_final.csv'

print('Path check:')
print(('  OK' if FILE_T2.exists() else '  NOT FOUND'), 'task2_subindustry_classification_final.csv')

Path check:
  OK task2_subindustry_classification_final.csv


In [2]:
# ── Load & Inspect ────────────────────────────────────────
t2 = pd.read_csv(FILE_T2, dtype={'SubIndustry': str, 'CompanyId': str})
t2['AsOfDate'] = pd.to_datetime(t2['AsOfDate'], errors='coerce')

print('=== Task 2 Raw Data ===')
print(f'Shape            : {t2.shape}')
print(f'Unique companies : {t2["CompanyId"].nunique():,}')
print(f'Unique SubIndustry: {t2["SubIndustry"].nunique()}')
print(f'Date range       : {t2["AsOfDate"].min().date()} to {t2["AsOfDate"].max().date()}')
print()

# Class distribution
dist = t2['SubIndustry'].value_counts()
print(f'Class distribution:')
print(f'  Most common  : {dist.index[0]} ({dist.iloc[0]:,})')
print(f'  Least common : {dist.index[-1]} ({dist.iloc[-1]})')
print(f'  Imbalance    : {dist.iloc[0] / dist.iloc[-1]:.0f}x')
print(f'  Classes < 5  : {(dist < 5).sum()}')
print(f'  Classes < 10 : {(dist < 10).sum()}')
print(f'  Classes = 1  : {(dist == 1).sum()}')
print()

# Text quality
t2['sd_empty'] = t2['SegmentDescription'].isna() | (t2['SegmentDescription'].str.strip() == '')
print(f'Empty SegmentDescription: {t2["sd_empty"].sum():,} ({t2["sd_empty"].mean()*100:.1f}%)')
print()
print('Sample rows:')
print(t2[['SegmentName', 'SegmentDescription', 'SubIndustry']].head(3).to_string())

=== Task 2 Raw Data ===
Shape            : (27537, 5)
Unique companies : 9,352
Unique SubIndustry: 428
Date range       : 2020-05-31 to 2024-12-31

Class distribution:
  Most common  : 3103001001 (2,018)
  Least common : 1014001003 (1)
  Imbalance    : 2018x
  Classes < 5  : 54
  Classes < 10 : 99
  Classes = 1  : 21

Empty SegmentDescription: 0 (0.0%)

Sample rows:
               SegmentName                                                                                                                                      SegmentDescription SubIndustry
0               Substrates  Substrates engages in the design, development, manufacture, and distribution of high-performance compound and single element semiconductor substrates.  3113001001
1  Raw Materials and Other                                               Raw Materials and Other pertains to the sale of raw materials integral to producing the substrate wafers.  3113001001
2      Frozen & Vegetables                                

In [ ]:
# ── Cleaning ──────────────────────────────────────────────
import re
try:
    import ftfy
    HAS_FTFY = True
except ImportError:
    HAS_FTFY = False
    print('ftfy not found — skipping encoding fix')

def normalize_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    if HAS_FTFY:
        text = ftfy.fix_text(str(text))
    else:
        text = str(text)
    text = re.sub(r'[“”‘’]', ' ', text)
    text = re.sub(r'\s*&\s*', ' and ', text)
    text = re.sub(r'[^ -]+', ' ', text)
    text = re.sub(r'\(\s*\)', ' ', text)
    text = ' '.join(text.split()).lower()
    return text.strip()

t2['SegmentName']        = t2['SegmentName'].apply(normalize_text)
t2['SegmentDescription'] = t2['SegmentDescription'].apply(normalize_text)

# Fill empty SegmentDescription with SegmentName only
# No LongProfile — excluded per friend's findings (hurts TF-IDF by 0.089)
t2['SegmentDescription'] = t2.apply(
    lambda row: row['SegmentName'] if not row['SegmentDescription'].strip() else row['SegmentDescription'],
    axis=1
)

# Build text input: SegmentName + SegmentDescription
# Repeat SegmentName for emphasis — same as Task 1 approach
t2['text_input'] = t2['SegmentName'] + ' ' + t2['SegmentName'] + ' ' + t2['SegmentDescription']

# Drop nulls
t2 = t2.dropna(subset=['SubIndustry']).copy()
t2 = t2[t2['text_input'].str.strip() != ''].copy()

print(f'Cleaned rows     : {len(t2):,}')
print(f'Empty text       : {t2["text_input"].str.strip().eq("").sum()}')
print()
print('Sample text_input:')
for i in range(3):
    row = t2.iloc[i]
    print(f'  [{row["SubIndustry"]}] {row["text_input"][:150]}')

In [ ]:
# ── GroupShuffleSplit on CompanyId ────────────────────────
le = LabelEncoder()
le.fit(t2['SubIndustry'])
t2['label']   = le.transform(t2['SubIndustry'])
NUM_CLASSES   = len(le.classes_)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(
    t2['text_input'], t2['SubIndustry'], groups=t2['CompanyId']
))

# Verify no company leakage
train_cos = set(t2.iloc[train_idx]['CompanyId'])
test_cos  = set(t2.iloc[test_idx]['CompanyId'])
assert len(train_cos & test_cos) == 0, 'Company leakage!'

# Save splits
np.savez_compressed(
    OUTPUT_DIR / 'canonical_splits.npz',
    t2_train_idx=train_idx,
    t2_test_idx=test_idx,
)

train_texts  = t2.iloc[train_idx]['text_input'].tolist()
test_texts   = t2.iloc[test_idx]['text_input'].tolist()
train_labels = t2.iloc[train_idx]['label'].values
test_labels  = t2.iloc[test_idx]['label'].values

# Class distribution in test
test_dist = pd.Series(test_labels).value_counts()
print(f'Split complete:')
print(f'  Train : {len(train_idx):,}  Test: {len(test_idx):,}')
print(f'  Company leakage: 0 classes in both splits')
print(f'  Test classes   : {test_dist.shape[0]} / {NUM_CLASSES}')
print(f'  Test zero-shot : {NUM_CLASSES - test_dist.shape[0]} classes never seen in test')
print(f'  Total classes  : {NUM_CLASSES}')

In [ ]:
# ── TF-IDF Vectorization ──────────────────────────────────
print('[1/3] Fitting TF-IDF vectorizers...')
t0 = time.time()

word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=100000,
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
)
char_vec = TfidfVectorizer(
    ngram_range=(3, 6),
    max_features=50000,
    sublinear_tf=True,
    min_df=3,
    analyzer='char_wb',
)

X_train_word = word_vec.fit_transform(train_texts)
X_train_char = char_vec.fit_transform(train_texts)
X_train      = hstack([X_train_word, X_train_char])

X_test_word  = word_vec.transform(test_texts)
X_test_char  = char_vec.transform(test_texts)
X_test       = hstack([X_test_word, X_test_char])

print(f'  word features : {X_train_word.shape[1]:,}')
print(f'  char features : {X_train_char.shape[1]:,}')
print(f'  total features: {X_train.shape[1]:,}')
print(f'  vectorize time: {time.time()-t0:.1f}s')

In [ ]:
# ── Train LinearSVC ───────────────────────────────────────
print('[2/3] Training LinearSVC...')
t0 = time.time()

clf = LinearSVC(
    C=0.5,
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
)
clf.fit(X_train, train_labels)
elapsed = time.time() - t0
print(f'  train time : {elapsed:.1f}s ({elapsed/60:.1f} min)')

# Also try LR C=5 (friend's config)
print('Training LR C=5 (friend config)...')
t0 = time.time()
clf_lr = LogisticRegression(
    C=5,
    max_iter=1000,
    class_weight='balanced',
    solver='saga',
    n_jobs=-1,
    random_state=42,
)
clf_lr.fit(X_train, train_labels)
elapsed = time.time() - t0
print(f'  LR train time: {elapsed:.1f}s ({elapsed/60:.1f} min)')

In [ ]:
# ── Evaluate Both ─────────────────────────────────────────
print('[3/3] Evaluating...')

def evaluate_model(clf, X_test, test_labels, name):
    y_pred   = clf.predict(X_test)
    y_true   = test_labels
    macro_f1 = float(f1_score(y_true, y_pred, average='macro',    zero_division=0))
    micro_f1 = float(f1_score(y_true, y_pred, average='micro',    zero_division=0))
    weighted = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    accuracy = float(accuracy_score(y_true, y_pred))
    per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    bottom50  = float(np.sort(per_class)[:50].mean())
    zero_shot = float((per_class == 0).mean())
    print()
    print(f'+-------------------------------------------------+')
    print(f'|  Task 2 Results — {name:28s}|')
    print(f'+-------------------------------------------------+')
    print(f'|  macro F1    : {macro_f1:.4f}                          |')
    print(f'|  micro F1    : {micro_f1:.4f}                          |')
    print(f'|  weighted F1 : {weighted:.4f}                          |')
    print(f'|  accuracy    : {accuracy:.4f}                          |')
    print(f'|  bottom-50   : {bottom50:.4f}                          |')
    print(f'|  zero F1 cls : {zero_shot:.3f} ({(per_class==0).sum()} classes)            |')
    print(f'+-------------------------------------------------+')
    return macro_f1, y_pred, per_class

macro_svc, y_pred_svc, pc_svc = evaluate_model(clf,    X_test, test_labels, 'LinearSVC C=0.5')
macro_lr,  y_pred_lr,  pc_lr  = evaluate_model(clf_lr, X_test, test_labels, 'LR C=5 (saga)  ')

# Pick best
if macro_lr > macro_svc:
    best_clf   = clf_lr
    best_pred  = y_pred_lr
    best_macro = macro_lr
    best_name  = 'LR C=5'
else:
    best_clf   = clf
    best_pred  = y_pred_svc
    best_macro = macro_svc
    best_name  = 'LinearSVC C=0.5'

print(f'\nBest model: {best_name}  macro={best_macro:.4f}')

In [ ]:
# ── Per-SubIndustry Analysis ──────────────────────────────
y_true    = test_labels
per_class = f1_score(y_true, best_pred, average=None, zero_division=0)
all_cls   = sorted(np.unique(np.concatenate([y_true, best_pred])))
pc_ser    = pd.Series(dict(zip(all_cls, per_class[all_cls]))).sort_values()

print('-- 20 Hardest SubIndustries --')
for cls_enc, f1_val in pc_ser.head(20).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_train = (train_labels == cls_enc).sum()
    n_test  = (y_true == cls_enc).sum()
    flag    = ' ZERO-SHOT' if n_test == 0 else (' sparse' if n_test < 5 else '')
    print(f'  {cls_str}  F1={f1_val:.3f}  train={n_train}  test={n_test}{flag}')

print()
print('-- 10 Easiest SubIndustries --')
for cls_enc, f1_val in pc_ser.tail(10).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_test  = (y_true == cls_enc).sum()
    print(f'  {cls_str}  F1={f1_val:.3f}  n_test={n_test}')

print()
print(f'Classes with F1 = 0    : {(per_class == 0).sum()}')
print(f'Classes with F1 < 0.5  : {(per_class < 0.5).sum()}')
print(f'Classes with F1 >= 0.75: {(per_class >= 0.75).sum()}')

In [ ]:
# ── C tuning — try different C values ─────────────────────
print('=== C Value Tuning ===')
print(f'{"C":>8}  {"Macro F1":>10}  {"Time":>8}')
print('-' * 35)

best_c_macro = 0.0
best_c       = 1.0
best_c_clf   = None

for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    t0 = time.time()
    c_clf = LogisticRegression(
        C=C, max_iter=500, class_weight='balanced',
        solver='saga', n_jobs=-1, random_state=42,
    )
    c_clf.fit(X_train, train_labels)
    y_c   = c_clf.predict(X_test)
    macro = float(f1_score(test_labels, y_c, average='macro', zero_division=0))
    elapsed = time.time() - t0
    flag = ' <- BEST' if macro > best_c_macro else ''
    print(f'{C:>8.1f}  {macro:>10.4f}  {elapsed:>7.1f}s{flag}', flush=True)
    if macro > best_c_macro:
        best_c_macro = macro
        best_c       = C
        best_c_clf   = c_clf

print(f'\nBest C={best_c}  macro={best_c_macro:.4f}')

In [ ]:
# ── Save Artifacts ────────────────────────────────────────
# Use best model from C tuning
final_clf   = best_c_clf
final_pred  = final_clf.predict(X_test)
final_macro = float(f1_score(test_labels, final_pred, average='macro', zero_division=0))
final_acc   = float(accuracy_score(test_labels, final_pred))

with open(ART_DIR / 'tfidf_word_vec.pkl', 'wb') as f:
    pickle.dump(word_vec, f)
with open(ART_DIR / 'tfidf_char_vec.pkl', 'wb') as f:
    pickle.dump(char_vec, f)
with open(ART_DIR / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(final_clf, f)
with open(ART_DIR / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

pd.DataFrame({
    'CompanyId'  : t2.iloc[test_idx]['CompanyId'].values,
    'SegmentName': t2.iloc[test_idx]['SegmentName'].values,
    'y_true'     : le.inverse_transform(test_labels),
    'y_pred'     : le.inverse_transform(final_pred),
    'correct'    : test_labels == final_pred,
}).to_csv(ART_DIR / 'task2_predictions.csv', index=False)

# Save cleaned data
t2.to_csv(OUTPUT_DIR / 'task2_gecs_cleaned.csv', index=False)

json.dump({
    'timestamp'  : datetime.now().isoformat(timespec='seconds'),
    'model'      : 'TF-IDF word(1-3)+char(3-6) + LR(C=' + str(best_c) + ')',
    'data'       : 'SegmentName + SegmentDescription, no LongProfile',
    'split'      : 'GroupShuffleSplit CompanyId 80/20',
    'n_train'    : int(len(train_idx)),
    'n_test'     : int(len(test_idx)),
    'n_classes'  : NUM_CLASSES,
    'macro_f1'   : round(final_macro, 4),
    'accuracy'   : round(final_acc, 4),
    'best_C'     : best_c,
}, open(ART_DIR / 'task2_summary.json', 'w'), indent=2)

print('All artifacts saved.')
print()
print('+----------------------------------------------+')
print('|  TASK 2 FINAL RESULT                        |')
print('+----------------------------------------------+')
print('|  Model  : TF-IDF word+char + LR             |')
print('|  Split  : GroupShuffleSplit (no leakage)    |')
print(f'|  macro F1: {final_macro:.4f}                          |')
print(f'|  accuracy: {final_acc:.4f}                          |')
print(f'|  Best C  : {best_c}                              |')
print('+----------------------------------------------+')